# Challenge 2 Setup

This notebook creates the schema and tables used in **Challenge 2: DAX to SQL Foundations**.

Covers only the foundational patterns from Demo 5 (aggregations, WHERE, CASE WHEN, JOINs, basic window functions).

In [0]:
import re

username = spark.sql("SELECT current_user()").first()[0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog_name = spark.sql("SELECT current_catalog()").first()[0]
schema_name = f"challenge_2_{clean_username}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}`")
spark.sql(f"USE CATALOG `{catalog_name}`")
spark.sql(f"USE SCHEMA `{schema_name}`")

print(f"\u2713 Schema ready: {catalog_name}.{schema_name}")

In [0]:
import random
from pyspark.sql import Row
from datetime import date, timedelta

random.seed(42)

# quarterly_sales - 12 months of monthly sales by region and category
# Simple, clean data for basic SQL pattern practice
regions = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Clothing', 'Home', 'Food']
channels = ['online', 'in_store']

rows = []
for month_offset in range(12):  # Jan-Dec 2024
    report_month = date(2024, month_offset + 1, 1)
    for region in regions:
        for category in categories:
            for channel in channels:
                base_rev = {'Electronics': 45000, 'Clothing': 30000, 'Home': 25000, 'Food': 20000}[category]
                region_factor = {'North': 1.2, 'South': 1.0, 'East': 1.1, 'West': 0.9}[region]
                channel_factor = 1.3 if channel == 'online' else 0.7
                revenue = round(base_rev * region_factor * channel_factor * random.uniform(0.8, 1.2) / 12, 2)
                units = int(revenue / random.uniform(30, 80))
                
                rows.append(Row(
                    report_month=report_month,
                    region=region,
                    product_category=category,
                    channel=channel,
                    revenue=revenue,
                    units_sold=units
                ))

df = spark.createDataFrame(rows)
df.write.mode("overwrite").saveAsTable("quarterly_sales")

count = spark.sql("SELECT COUNT(*) FROM quarterly_sales").first()[0]
print(f"\u2713 Table created: quarterly_sales ({count} rows - 12 months \u00d7 4 regions \u00d7 4 categories \u00d7 2 channels)")

In [0]:
from pyspark.sql import Row

# category_targets - annual revenue targets per category (for JOINs)
targets = [
    Row(product_category='Electronics', annual_target=500000.0, priority='High'),
    Row(product_category='Clothing', annual_target=350000.0, priority='Medium'),
    Row(product_category='Home', annual_target=280000.0, priority='Medium'),
    Row(product_category='Food', annual_target=220000.0, priority='Low'),
]

df_targets = spark.createDataFrame(targets)
df_targets.write.mode("overwrite").saveAsTable("category_targets")

print(f"\u2713 Table created: category_targets (4 rows - annual targets per category)")

In [0]:
print("")
print("="*60)
print("\u2713 CHALLENGE 2 SETUP COMPLETE")
print("="*60)
print(f"")
print(f"Catalog:  {catalog_name}")
print(f"Schema:   {schema_name}")
print(f"")
print(f"Tables:")
print(f"  \u2022 quarterly_sales   - 384 rows (12 months \u00d7 4 regions \u00d7 4 categories \u00d7 2 channels)")
print(f"  \u2022 category_targets   - 4 rows (annual targets per category)")